# Manual RAG Pipeline: Mechanisms First

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline from scratch.
You'll see every step explicitly before we move to frameworks like LangChain.

**Works on:** Google Colab, Local Jupyter (Mac/Windows/Linux)

**Pipeline Overview:**
```
Documents → Chunking → Embedding → Index (FAISS)
                                        ↓
User Query → Embed Query → Similarity Search → Top-K Chunks
                                                    ↓
                                        Prompt Assembly → LLM → Answer
```

## TODO — Topic 5 RAG Course Project Checklist

- **Exercise 0:** Set-up — Get notebook running; unzip Corpora.zip. Use PDFs from `Corpora/<corpus>/pdf_embedded/`.
- **Exercise 1:** Open model RAG vs no RAG — Compare Qwen 2.5 1.5B with/without RAG on Model T manual and Congressional Record.
- **Exercise 2:** Open model + RAG vs large model — Run GPT-4o Mini with no tools on same queries.
- **Exercise 3:** Open model + RAG vs frontier chat — Compare local Qwen+RAG vs GPT-4/Claude (web).
- **Exercise 4:** Effect of top-K — Test k = 1, 3, 5, 10, 20.
- **Exercise 5:** Unanswerable questions — Off-topic, related-but-missing, false premise.
- **Exercise 6:** Query phrasing sensitivity — Same question in 5+ phrasings.
- **Exercise 7:** Chunk overlap — Re-chunk with overlap 0, 64, 128, 256.
- **Exercise 8:** Chunk size — Chunk at 128, 256, 512, 1024, 2048.
- **Exercise 9:** Retrieval score analysis — 10 queries, top-10 chunks, score distribution.
- **Exercise 10:** Prompt template variations — Minimal, strict grounding, citation, permissive, structured.
- **Exercise 11:** Failure mode catalog — Computation, temporal, comparison, ambiguous, multi-hop, etc.
- **Exercise 12:** Cross-document synthesis — Questions needing multiple chunks.

## Setup

First, let's install the required packages and detect our compute environment.

In [1]:
# Install dependencies
# On Colab, these install quickly. Locally, you may already have them.
# Use a kernel-aware install when available; fall back to subprocess otherwise.
try:
    ip = get_ipython()
    ip.run_line_magic('pip', 'install -q torch transformers sentence-transformers faiss-cpu pymupdf accelerate ipyfilechooser')
except NameError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'transformers', 'sentence-transformers', 'faiss-cpu', 'pymupdf', 'accelerate', 'ipyfilechooser'])
# For Exercise 2 (GPT-4o Mini): add 'openai' to the list above if needed


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.1 MB/s eta 0:00:00


In [2]:
# =============================================================================
# ENVIRONMENT AND DEVICE DETECTION
# =============================================================================
import os
import sys

# Enable MPS fallback for any PyTorch operations not yet implemented on Metal
# This MUST be set before importing torch
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# Prevent kernel crash from duplicate OpenMP libraries (PyTorch + FAISS conflict on macOS)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
from typing import Tuple

def detect_environment() -> str:
    """Detect if we're running on Colab or locally."""
    try:
        import google.colab
        return 'colab'
    except ImportError:
        return 'local'

def get_device() -> Tuple[str, torch.dtype]:
    """
    Detect the best available compute device.

    Priority: CUDA > MPS (Apple Silicon) > CPU

    Returns:
        Tuple of (device_string, recommended_dtype)

    Notes:
        - CUDA: Use float16 for memory efficiency (Tensor Cores optimize this)
        - MPS: Use float32 - Apple Silicon doesn't have the same float16
               optimizations as NVIDIA, and float32 is often faster
        - CPU: Use float32 (float16 not well supported on CPU)
    """
    if torch.cuda.is_available():
        device = 'cuda'
        dtype = torch.float16
        device_name = torch.cuda.get_device_name(0)
        memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✓ Using CUDA GPU: {device_name} ({memory_gb:.1f} GB)")

    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        device = 'mps'
        dtype = torch.float32  # float32 is often faster on Apple Silicon!
        print("✓ Using Apple Silicon GPU (MPS)")
        print("  Note: Using float32 (faster than float16 on Apple Silicon)")

    else:
        device = 'cpu'
        dtype = torch.float32
        print("⚠ Using CPU (no GPU detected)")
        print("  Tip: For faster processing, use a machine with a GPU")

    return device, dtype

# Detect environment and device
ENVIRONMENT = detect_environment()
DEVICE, DTYPE = get_device()

print(f"\nEnvironment: {ENVIRONMENT.upper()}")
print(f"Device: {DEVICE}, Dtype: {DTYPE}")

✓ Using CUDA GPU: Tesla T4 (15.6 GB)

Environment: COLAB
Device: cuda, Dtype: torch.float16


## Load Your Documents

**Cell 1:** Configure your document source and select/upload files
- **Local Jupyter**: Use the folder picker, then run Cell 2
- **Colab + Upload**: Files upload immediately (blocking), then run Cell 2
- **Colab + Drive**: Set `USE_GOOGLE_DRIVE = True`, mounts Drive and shows picker, then run Cell 2

**Cell 2:** Confirms selection and lists documents

In [3]:
# =============================================================================
# CELL 1: SELECT DOCUMENT SOURCE
# =============================================================================
# This cell either:
#   - Shows a folder picker (Local or Colab+Drive) - NON-BLOCKING
#   - Shows an upload dialog (Colab+Upload) - BLOCKING
#
# If a folder picker is shown, SELECT YOUR FOLDER BEFORE running Cell 2.
# The picker widget is non-blocking, so the code continues before you select.
# =============================================================================

from pathlib import Path

# ------------- COLAB USERS: CONFIGURE HERE -------------
USE_GOOGLE_DRIVE = False  # Set to True to use Google Drive instead of uploading
# -------------------------------------------------------

# Default folder: use Corpora from course project (unzip Corpora.zip first).
_folder_default = Path("Corpora/ModelTService")
DOC_FOLDER = str(_folder_default) if _folder_default.exists() else "documents"
folder_chooser = None  # Will hold the picker widget if used

if ENVIRONMENT == 'colab':
    if USE_GOOGLE_DRIVE:
        # ----- COLAB + GOOGLE DRIVE -----
        # Mount Drive first, then show folder picker
        from google.colab import drive
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
        print("✓ Google Drive mounted\n")

        # Now show folder picker for the Drive
        try:
            from ipyfilechooser import FileChooser

            folder_chooser = FileChooser(
                path='/content/drive/MyDrive',
                title='Select your documents folder in Google Drive',
                show_only_dirs=True,
                select_default=True
            )
            print("📁 Select your documents folder below, then run Cell 2:")
            print("   (The picker is non-blocking - select BEFORE running the next cell)")
            display(folder_chooser)

        except ImportError:
            # Fallback: manual path entry
            print("Folder picker not available.")
            print("Edit DOC_FOLDER below with your Google Drive path, then run Cell 2:")
            DOC_FOLDER = '/content/drive/MyDrive/your_documents_folder'  # ← Edit this!
            print(f"  DOC_FOLDER = '{DOC_FOLDER}'")
    else:
        # ----- COLAB + UPLOAD -----
        # Upload dialog blocks until complete, so DOC_FOLDER is ready when done
        from google.colab import files
        os.makedirs(DOC_FOLDER, exist_ok=True)

        print("Upload your documents (PDF, TXT, or MD):")
        print("(This dialog blocks until upload is complete)\n")
        uploaded = files.upload()

        for filename in uploaded.keys():
            os.rename(filename, f'{DOC_FOLDER}/{filename}')
            print(f"  ✓ Saved: {DOC_FOLDER}/{filename}")

        print(f"\n✓ Upload complete. Run Cell 2 to continue.")

else:
    # ----- LOCAL JUPYTER -----
    # Show folder picker
    print("Running locally\n")

    try:
        from ipyfilechooser import FileChooser

        folder_chooser = FileChooser(
            path=str(Path.home()),
            title='Select your documents folder',
            show_only_dirs=True,
            select_default=True
        )
        print("📁 Select your documents folder below, then run Cell 2:")
        print("   (The picker is non-blocking - select BEFORE running the next cell)")
        display(folder_chooser)

    except ImportError:
        # Fallback: manual path entry
        print("Folder picker not available (ipyfilechooser not installed).")
        print(f"\nUsing default folder: {Path(DOC_FOLDER).absolute()}")
        print("\nTo use a different folder, edit DOC_FOLDER in this cell:")
        print("  DOC_FOLDER = '/path/to/your/documents'")
        os.makedirs(DOC_FOLDER, exist_ok=True)

Upload your documents (PDF, TXT, or MD):
(This dialog blocks until upload is complete)



Saving ModelTNew.pdf to ModelTNew.pdf
  ✓ Saved: documents/ModelTNew.pdf

✓ Upload complete. Run Cell 2 to continue.


In [4]:
# =============================================================================
# CELL 2: CONFIRM SELECTION AND LIST DOCUMENTS
# =============================================================================
# If you used a folder picker above, make sure you selected a folder
# BEFORE running this cell. The picker is non-blocking.
# =============================================================================

# Read selection from folder picker (if one was used)
if folder_chooser is not None and folder_chooser.selected_path:
    DOC_FOLDER = folder_chooser.selected_path
    print(f"✓ Using selected folder: {DOC_FOLDER}")
elif folder_chooser is not None:
    print("⚠ No folder selected in picker!")
    print("  Please go back to Cell 1, select a folder, then run this cell again.")
else:
    # No picker used (upload or manual path)
    print(f"✓ Using folder: {DOC_FOLDER}")

# Confirm folder (listing skipped for speed)
doc_path = Path(DOC_FOLDER)
if doc_path.exists():
    print(f"✓ Folder set: {doc_path.absolute()}")
    print("  Run the next cells to load, chunk, and index documents.")
else:
    print(f"⚠ Folder not found: {DOC_FOLDER}")
    print("  Please set DOC_FOLDER in the previous cell and run it again.")

✓ Using folder: documents
✓ Folder set: /content/documents
  Run the next cells to load, chunk, and index documents.


---
## Stage 1: Document Loading

We need to extract text from our documents. For PDFs with embedded text,
PyMuPDF (fitz) reads the text layer directly - no OCR needed.

**Corpora:** Use PDFs from `Corpora/<name>/pdf_embedded/`. The `.txt` files in `txt/` are for checking retrieval vs OCR issues.

In [5]:
# Exercise 1 (and reuse): Official query lists. Reference: CR Jan 13, 20, 21, 23, 2026.
QUERIES_MODEL_T = [
    "How do I adjust the carburetor on a Model T?",
    "What is the correct spark plug gap for a Model T Ford?",
    "How do I fix a slipping transmission band?",
    "What oil should I use in a Model T engine?",
]
QUERIES_CR = [
    "What did Mr. Flood have to say about Mayor David Black in Congress on January 13, 2026?",
    "What mistake did Elise Stefanik make in Congress on January 23, 2026?",
    "What is the purpose of the Main Street Parity Act?",
    "Who in Congress has spoken for and against funding of pregnancy centers?",
]

In [6]:
import fitz  # PyMuPDF
from typing import List, Tuple

def load_text_file(filepath: str) -> str:
    """Load a plain text file."""
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()


def load_pdf_file(filepath: str) -> str:
    """
    Extract text from a PDF with embedded text.

    PyMuPDF reads the text layer directly.
    For scanned PDFs without embedded text, you'd need OCR.
    """
    doc = fitz.open(filepath)
    text_parts = []

    for page_num, page in enumerate(doc):
        text = page.get_text()
        if text.strip():
            # Add page marker for debugging/citation
            text_parts.append(f"\n[Page {page_num + 1}]\n{text}")

    doc.close()
    return "\n".join(text_parts)


def load_documents(doc_folder: str) -> List[Tuple[str, str]]:
    """Load all documents from a folder. Returns list of (filename, content)."""
    documents = []
    folder = Path(doc_folder)

    for filepath in folder.rglob("*"):
        try:
            if not filepath.is_file():
                continue
        except OSError:
            continue
        if filepath.suffix.lower() not in ('.pdf', '.txt', '.md', '.text'):
            continue
        try:
            if filepath.suffix.lower() == '.pdf':
                content = load_pdf_file(str(filepath))
            elif filepath.suffix.lower() in ['.txt', '.md', '.text']:
                content = load_text_file(str(filepath))
            else:
                continue

            if content.strip():
                documents.append((filepath.name, content))
                print(f"✓ Loaded: {filepath.name} ({len(content):,} chars)")
        except Exception as e:
            print(f"✗ Error loading {filepath}: {e}")

    return documents

In [7]:
# Load your documents
documents = load_documents(DOC_FOLDER)
print(f"\nLoaded {len(documents)} documents")

if len(documents) == 0:
    print("\n⚠ No documents loaded! Please add PDF or TXT files to the documents folder.")

✓ Loaded: ModelTNew.pdf (469,891 chars)

Loaded 1 documents


In [8]:
# Inspect a document to verify loading worked
if documents:
    filename, content = documents[0]
    print(f"First document: {filename}")
    print(f"Total length: {len(content):,} characters")
    print(f"\nFirst 1000 characters:\n{'-'*40}")
    print(content[:1000])

First document: ModelTNew.pdf
Total length: 469,891 characters

First 1000 characters:
----------------------------------------

[Page 2]
S E R V I  
Detailed Instructions for 
Servicing Ford Gars 
PRICE $250 
Published by 
DETROIT, MICHIGAN, U. S. A. 


[Page 3]
Contents 
Foreword . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .  
111 
. . .  
Essentials of good service. 
: . . . . .  : . . . . . . . . . . . . . . . . . . . . . . .  
ix 
. . . . . . . . . . . . . . . . . . . .  
Ideal shop layout for average size dealer. 
x 
Essential shop equipment. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .  
xi 
... 
The parts department. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .  
xi11 
An attractive parts department. . . . . . . . . . . . . . . . . . . . . . . . . . . . .  xiv 
Service follow-up system. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .  
xv 
CHAPTER I 
Disassembli

---
## Stage 2: Chunking

Documents need to be split into pieces small enough to be relevant but large enough to carry meaning.

**Why overlap?** If a key sentence sits right at a chunk boundary, splitting without overlap might cut it in half. Overlap ensures that information near boundaries appears intact in at least one chunk.

**Experiment:** Try different chunk sizes (256, 512, 1024) and see how it affects retrieval!

In [9]:
from dataclasses import dataclass

@dataclass
class Chunk:
    """A chunk of text with metadata for tracing back to source."""
    text: str
    source_file: str
    chunk_index: int
    start_char: int
    end_char: int


def chunk_text(
    text: str,
    source_file: str,
    chunk_size: int = 512,
    chunk_overlap: int = 128
) -> List[Chunk]:
    """
    Split text into overlapping chunks.

    We try to break at sentence or paragraph boundaries
    to avoid cutting mid-thought.
    """
    chunks = []
    start = 0
    chunk_index = 0

    while start < len(text):
        end = start + chunk_size

        # Try to break at a good boundary
        if end < len(text):
            # Look for paragraph break first
            para_break = text.rfind('\n\n', start + chunk_size // 2, end)
            if para_break != -1:
                end = para_break + 2
            else:
                # Look for sentence break
                sentence_break = text.rfind('. ', start + chunk_size // 2, end)
                if sentence_break != -1:
                    end = sentence_break + 2

        chunk_text_str = text[start:end].strip()

        if chunk_text_str:
            chunks.append(Chunk(
                text=chunk_text_str,
                source_file=source_file,
                chunk_index=chunk_index,
                start_char=start,
                end_char=end
            ))
            chunk_index += 1

        # Move forward, accounting for overlap
        start = end - chunk_overlap
        if chunks and start <= chunks[-1].start_char:
            start = end  # Safety: ensure progress

    return chunks

In [10]:
# ============================================
# EXPERIMENT: Try different chunk sizes!
# ============================================
CHUNK_SIZE = 512      # Try: 256, 512, 1024
CHUNK_OVERLAP = 128   # Try: 64, 128, 256
# For Ex 7/8 use rebuild_pipeline() — see cell after FAISS index.

# Chunk all documents
all_chunks = []
for filename, content in documents:
    doc_chunks = chunk_text(content, filename, CHUNK_SIZE, CHUNK_OVERLAP)
    all_chunks.extend(doc_chunks)
    print(f"{filename}: {len(doc_chunks)} chunks")

print(f"\nTotal: {len(all_chunks)} chunks")

ModelTNew.pdf: 1496 chunks

Total: 1496 chunks


In [11]:
# Inspect some chunks
if all_chunks:
    print("Sample chunks:")
    indices_to_show = [0, len(all_chunks)//2, -1] if len(all_chunks) > 2 else range(len(all_chunks))
    for i in indices_to_show:
        chunk = all_chunks[i]
        print(f"\n{'='*60}")
        print(f"Chunk {chunk.chunk_index} from {chunk.source_file}")
        print(f"{'='*60}")
        print(chunk.text[:300] + "..." if len(chunk.text) > 300 else chunk.text)

Sample chunks:

Chunk 0 from ModelTNew.pdf
[Page 2]
S E R V I  
Detailed Instructions for 
Servicing Ford Gars 
PRICE $250 
Published by 
DETROIT, MICHIGAN, U. S. A. 


[Page 3]
Contents 
Foreword . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .  
111 
. . .  
Essentials of good service. 
: . . ...

Chunk 748 from ModelTNew.pdf
driving it out of keyway with a small 
hammer and drift (See Fig. 318). With a fine file dress off any rough 
edges on keyway. 
632 
R emove the old felt retainer by forcing it off end of housing 
with a screw driver. 
(See Fig. 319). 
633 
Withdraw outer roller bearing by inserting a small wire hoo...

Chunk 1495 from ModelTNew.pdf
tening.. . . ....... . . . . .. . .. . .. 
74 
removing . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 
57 
Wiring diagram, cars equipped with starter (Page 19) 
not equipped with starter (Page 20) 
improved cars equipped with starter (Page 287) · 
not equipped w

---
## Stage 3: Embedding

Embeddings map text to dense vectors where **semantic similarity = geometric proximity**.

A sentence about "cardiac arrest" and one about "heart attack" will have similar embeddings even though they share no words.

**Note:** sentence-transformers does NOT auto-detect Apple MPS - we must pass the device explicitly.

In [12]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
# Options:
# - "sentence-transformers/all-MiniLM-L6-v2": Fast, small (80MB), good quality
# - "BAAI/bge-small-en-v1.5": Better for retrieval, similar size

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {EMBEDDING_MODEL}")
print(f"Device: {DEVICE}")

# Must explicitly pass device for MPS support!
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
EMBEDDING_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {EMBEDDING_DIM}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


In [13]:
# DEMO: See how embeddings capture semantic similarity
test_sentences = [
    "The engine needs regular oil changes.",
    "Motor oil should be replaced periodically.",
    "The Senate convened at noon.",
    "Congress began its session at midday."
]

test_embeddings = embed_model.encode(test_sentences)

# Compute cosine similarity matrix
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print("Cosine similarity matrix:")
print("\n" + " " * 40 + "  [0]    [1]    [2]    [3]")
for i, s1 in enumerate(test_sentences):
    sims = [cosine_sim(test_embeddings[i], test_embeddings[j]) for j in range(4)]
    print(f"[{i}] {s1[:35]:35} {sims[0]:.3f}  {sims[1]:.3f}  {sims[2]:.3f}  {sims[3]:.3f}")

print("\n→ Notice: [0]-[1] are similar (both about oil), [2]-[3] are similar (both about Congress)")

Cosine similarity matrix:

                                          [0]    [1]    [2]    [3]
[0] The engine needs regular oil change 1.000  0.728  -0.045  -0.032
[1] Motor oil should be replaced period 0.728  1.000  0.014  0.035
[2] The Senate convened at noon.        -0.045  0.014  1.000  0.684
[3] Congress began its session at midda -0.032  0.035  0.684  1.000

→ Notice: [0]-[1] are similar (both about oil), [2]-[3] are similar (both about Congress)


In [14]:
# Embed all chunks - this may take a few minutes for large corpora
if all_chunks:
    print(f"Embedding {len(all_chunks)} chunks on {DEVICE}...")
    chunk_texts = [c.text for c in all_chunks]
    chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=True)
    chunk_embeddings = chunk_embeddings.astype('float32')  # FAISS wants float32
    print(f"Embeddings shape: {chunk_embeddings.shape}")
else:
    print("No chunks to embed - please load documents first.")

Embedding 1496 chunks on cuda...


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Embeddings shape: (1496, 384)


---
## Stage 4: Vector Index (FAISS)

FAISS efficiently finds nearest neighbors in high-dimensional spaces.

We use a simple **flat index** (brute-force search) which is transparent and works well for up to ~100k vectors. For larger corpora, you'd use approximate methods like IVF or HNSW.

**Note:** FAISS GPU support is CUDA-only. On MPS/CPU, we use faiss-cpu (still very fast for <100k vectors).

In [15]:
import faiss

# Create FAISS index
# IndexFlatIP = Inner Product (for cosine similarity on normalized vectors)
index = faiss.IndexFlatIP(EMBEDDING_DIM)

if all_chunks:
    # Normalize vectors so inner product = cosine similarity
    faiss.normalize_L2(chunk_embeddings)

    # Add vectors to index
    index.add(chunk_embeddings)
    print(f"Index built with {index.ntotal} vectors")
else:
    print("No embeddings to index - please load and embed documents first.")

Index built with 1496 vectors


---
## Stage 5: Retrieval

Now we can search! Given a query, we:
1. Embed the query with the same model
2. Find the top-k most similar chunks
3. Return those chunks as context

In [16]:
# Helper for Exercises 7 & 8: rebuild chunks + index with different chunk_size / chunk_overlap.
def rebuild_pipeline(chunk_size: int = 512, chunk_overlap: int = 128):
    """Re-chunk documents, re-embed, and rebuild FAISS index. Updates global all_chunks and index."""
    global all_chunks, index
    all_chunks = []
    for filename, content in documents:
        all_chunks.extend(chunk_text(content, filename, chunk_size=chunk_size, chunk_overlap=chunk_overlap))
    chunk_embeddings = embed_model.encode([c.text for c in all_chunks], show_progress_bar=True).astype("float32")
    faiss.normalize_L2(chunk_embeddings)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(chunk_embeddings)
    print(f"Rebuilt: {len(all_chunks)} chunks, chunk_size={chunk_size}, chunk_overlap={chunk_overlap}")

In [17]:
def retrieve(query: str, top_k: int = 5):
    """
    Retrieve the top-k most relevant chunks for a query.

    Returns: List of (chunk, similarity_score) tuples
    """
    # Embed the query
    query_embedding = embed_model.encode([query]).astype('float32')
    faiss.normalize_L2(query_embedding)

    # Search
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            results.append((all_chunks[idx], float(score)))

    return results

In [18]:
# Test retrieval
# ============================================
# TRY DIFFERENT QUERIES FOR YOUR CORPUS!
# ============================================
test_query = "What is the procedure for engine maintenance?"  # ← Modify this!

if index.ntotal > 0:
    results = retrieve(test_query, top_k=5)

    print(f"Query: {test_query}\n")
    print("Top 5 retrieved chunks:")
    for i, (chunk, score) in enumerate(results, 1):
        print(f"\n[{i}] Score: {score:.4f} | Source: {chunk.source_file}")
        print(f"    {chunk.text[:200]}...")
else:
    print("Index is empty - please load, chunk, and embed documents first.")

Query: What is the procedure for engine maintenance?

Top 5 retrieved chunks:

[1] Score: 0.5769 | Source: ModelTNew.pdf
    en a car is brought in for major repair 
work is to first assign the car to a section of the shop set aside for re- 
pair jobs. The assembly to be overhauled is then removed from the 
car and by means...

[2] Score: 0.5432 | Source: ModelTNew.pdf
    cleaning. After the cleaning operation it is trans- 
ferred to the stand or repair bench on which the work is to be performed. 
When the overhaul work is completed the assembly is returned by 
means o...

[3] Score: 0.5160 | Source: ModelTNew.pdf
    n cock in bottom of radiator and fill radiator with 
clean water. 
359 
Time Study 
Engine and Transmission Overhaul 
Hrs. Min. 
1 
Install car covers on front fenders, running board, steering 
wheel ...

[4] Score: 0.4982 | Source: ModelTNew.pdf
    he job) 
Hrs. 
Drain water, install car covers, remove hood and cylinder 
head . ..... .... .. .... . ..... .. . . ...... 

---
## Stage 6: Generation (LLM)

Now we load a local LLM to generate answers from the retrieved context.

**Recommended models:**
- `Qwen/Qwen2.5-1.5B-Instruct` - Best instruction following at this size
- `Qwen/Qwen2.5-3B-Instruct` - Even better if you have 8GB+ VRAM
- `meta-llama/Llama-3.2-1B-Instruct` - Alternative, slightly weaker

**Device handling:**
- CUDA: Uses `device_map="auto"` and float16
- MPS: Loads to CPU first, then moves to MPS with float32
- CPU: Uses float32 (slower but works)

In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# ============================================
# CHOOSE YOUR MODEL
# ============================================
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # Or try "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading LLM: {LLM_MODEL}")
print(f"Device: {DEVICE}, Dtype: {DTYPE}")
print("This may take a few minutes on first run...\n")

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

# Load with appropriate settings for each device type
if DEVICE == 'cuda':
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        device_map="auto",
        torch_dtype=DTYPE,
        trust_remote_code=True
    )
    print("Model loaded on CUDA")

elif DEVICE == 'mps':
    # For MPS, load to CPU first, then move to MPS
    # (device_map="auto" doesn't work well with MPS)
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        torch_dtype=DTYPE,
        trust_remote_code=True
    )
    model = model.to(DEVICE)
    print("Model loaded on MPS (Apple Silicon)")

else:
    # CPU
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        torch_dtype=DTYPE,
        trust_remote_code=True
    )
    print("Model loaded on CPU (this will be slow)")

Loading LLM: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda, Dtype: torch.float16
This may take a few minutes on first run...



config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on CUDA


In [20]:
def generate_response(prompt: str, max_new_tokens: int = 512, temperature: float = 0.3) -> str:
    """
    Generate a response from the LLM.

    Lower temperature = more focused/deterministic
    Higher temperature = more creative/random
    """
    inputs = tokenizer(prompt, return_tensors="pt")

    # Move inputs to the correct device
    if DEVICE == 'cuda':
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    else:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True if temperature > 0 else False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the new tokens
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()

---
## Stage 7: The Complete RAG Pipeline

Now we put it all together. The **prompt template** is critical - it must instruct the model to use the retrieved context.

In [21]:
# The RAG prompt template
PROMPT_TEMPLATE = """You are a helpful assistant that answers questions based on the provided context.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer the question based ONLY on the information in the context above
- If the context doesn't contain enough information to answer, say so
- Quote relevant parts of the context to support your answer
- Be concise and direct

ANSWER:"""


def direct_query(question: str, max_new_tokens: int = 512) -> str:
    """Ask the LLM directly with no retrieved context (for RAG vs no-RAG comparison)."""
    prompt = f"""Answer this question:
{question}

Answer:"""
    return generate_response(prompt, max_new_tokens=max_new_tokens)

def rag_query(question: str, top_k: int = 5, show_context: bool = False, prompt_template: str = None) -> str:
    """The complete RAG pipeline. prompt_template: custom template for Exercise 10."""
    # Step 1: Retrieve
    results = retrieve(question, top_k)

    # Format context
    context_parts = []
    for chunk, score in results:
        context_parts.append(f"[Source: {chunk.source_file}, Relevance: {score:.3f}]\n{chunk.text}")
    context = "\n\n---\n\n".join(context_parts)

    if show_context:
        print("=" * 60)
        print("RETRIEVED CONTEXT:")
        print("=" * 60)
        print(context)
        print("=" * 60 + "\n")

    # Step 2: Build prompt (use custom template if provided)
    template = prompt_template if prompt_template is not None else PROMPT_TEMPLATE
    prompt = template.format(context=context, question=question)

    # Step 3: Generate
    answer = generate_response(prompt)

    return answer

In [22]:
# ============================================
# TEST YOUR RAG PIPELINE!
# ============================================

question = "What maintenance is required for the engine?"  # ← Modify for your corpus!

if index.ntotal > 0:
    print(f"Question: {question}\n")
    print("Generating answer...\n")

    answer = rag_query(question, top_k=5, show_context=True)

    print("ANSWER:")
    print(answer)
else:
    print("Pipeline not ready - please complete all previous stages first.")

Question: What maintenance is required for the engine?

Generating answer...

RETRIEVED CONTEXT:
[Source: ModelTNew.pdf, Relevance: 0.496]
he job) 
Hrs. 
Drain water, install car covers, remove hood and cylinder 
head . ..... .... .. .... . ..... .. . . ...... .. .. .... ..... . 
Remove crankcase cover, connecting rod caps and pistons .. 
Fit new pistons, pins, assemble connecting rods and check 
I for alignment, fit and install piston rings ..... ' . . . . . . . . . 1 
Install pistons and connecting rod caps ... . . . . ..... .. .. . 
Install crankcase cover, clean and install cylinder head . .. .

---

[Source: ModelTNew.pdf, Relevance: 0.493]
and fill radiator 
with clean water. 
499 
Time Study 
Oil Leak at Front End of Crankshaft 
(One man doing the job) 
Hrs . Min. 
1 
Drain water, install car covers, remove radiator, fan and 
front cover. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 
28 
2 
Change felts in crankcase and cover, install cover, fan and 
radia

---
## Experiments: Understanding RAG Behavior

Now that you have a working pipeline, try these experiments to understand how each component affects the results.

In [23]:
# EXPERIMENT 1: Compare WITH vs WITHOUT RAG
# ==========================================

question = QUERIES_MODEL_T
#question = "What are the specifications for the landing gear?"  # ← Use a corpus-specific question!

if index.ntotal > 0:
    # WITHOUT RAG - just ask the model directly
    direct_prompt = f"""Answer this question:
{question}

Answer:"""

    print("WITHOUT RAG (model's own knowledge):")
    print("-" * 40)
    direct_answer = generate_response(direct_prompt)
    print(direct_answer)

    print("\n" + "=" * 60 + "\n")

    # WITH RAG
    print("WITH RAG (using retrieved context):")
    print("-" * 40)
    rag_answer = rag_query(question, top_k=5)
    print(rag_answer)
else:
    print("Please complete the pipeline setup first.")

WITHOUT RAG (model's own knowledge):
----------------------------------------
The first two questions are about adjusting the carburetor and spark plugs, while the last one is about fixing a slipping transmission band. To answer these questions:

1. Adjusting the carburetor on a Model T involves turning the adjustment screw located at the top of the carburetor to increase or decrease fuel flow. This can be done by turning the screw clockwise to add more fuel (increase speed) or counterclockwise to reduce fuel (decrease speed).

2. For the spark plug gap, it's typically 0.035 inches (about 9/64ths of an inch). You may need to remove the spark plug wire from the distributor cap and measure the distance between the center electrode and the side electrode with a feeler gauge.

3. Fixing a slipping transmission band requires replacing the faulty band. First, you'll need to disassemble the transmission housing and locate the slipping band. Once found, replace it with a new one that matches t

In [31]:
# EXPERIMENT 2: Effect of top_k
# ==========================================

question = "What is the correct spark plug gap for a Model T Ford?"  # ← Use a corpus-specific question!

if index.ntotal > 0:
    for k in [1, 3, 5, 10, 20]:
        print(f"\n{'='*60}")
        print(f"TOP_K = {k}")
        print(f"{'='*60}")
        answer = rag_query(question, top_k=k)
        print(answer[:500] + "..." if len(answer) > 500 else answer)
else:
    print("Please complete the pipeline setup first.")


TOP_K = 1
The context does not provide specific details about the correct spark plug gap for a Model T Ford. It only mentions that if a spark occurs between the spark plug wire and the cylinder, the problem lies in the commutator or commutator loom, which can be corrected by inspecting the loom for breaks in the wire and insulation and noting where they occur. To determine the correct gap size, one would need to refer to the official specifications or guidelines for Model T Ford spark plugs as published b...

TOP_K = 3
Based on the information provided in the context, there isn't specific mention of the recommended spark plug gap for a Model T Ford. However, it does state that if a spark occurs between the spark plug wire and the cylinder, the problem lies in the commutator or commutator loom and can be corrected by inspecting the loom for breaks in the wire and insulation and noting where necessary. This suggests that proper inspection and repair procedures should be followed rather 

In [32]:
# EXPERIMENT 3: Question the corpus CAN'T answer(Modified)

unanswerable_questions = [
    "What is the capital of France?",
    "What's the horsepower of a 1925 Model T?",
    "Why does the manual recommend synthetic oil?"
]

MODIFIED_PROMPT_TEMPLATE = """You are a helpful assistant that answers questions based on the provided context.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer the question based ONLY on the information in the context above
- If the context doesn't contain the answer, say 'I cannot answer this from the available documents.'
- Quote relevant parts of the context to support your answer
- Be concise and direct

ANSWER:"""

if index.ntotal > 0:
    for q in unanswerable_questions:
        print(f"Question: {q}\n")
        answer = rag_query(q, top_k=5, show_context=True)
        print(f"\nAnswer: {answer}\n")
        print("=" * 60)

    for q in unanswerable_questions:
        print(f"Question (Modified Prompt): {q}\n")
        answer = rag_query(q, top_k=5, show_context=True, prompt_template=MODIFIED_PROMPT_TEMPLATE)
        print(f"\nAnswer: {answer}\n")
        print("=" * 60)
else:
    print("Please complete the pipeline setup first.")

Question: What is the capital of France?

RETRIEVED CONTEXT:
[Source: ModelTNew.pdf, Relevance: 0.133]
cessary use the dent puller as explained in Par. 828. 
Removing Top Tank Top 
833 
The top tank top can be removed in two ways. The first method 
is used when the old top tank top is 0. K. and is to be used again. 
The second method is used when the top section of the tank is beyond 
repair. 
First Method 
834 
To remove the top by the first method, remove the wall as de-
scribed in Pars. 824, 825 or 829. 
835 
Remove the splash plate and the overflow pipe as described in 
Par. 843.

---

[Source: ModelTNew.pdf, Relevance: 0.124]
ccess to the sediment bulb and simplifies the operation 
of removing water and foreign matter which the sediment bulb collects. 
1230 The tank is filled from the outside of the car, the filler cap being 
located in the center of the cowl underneath a rain proof cover. A 
large trough or wall built around the filler and connected to an overflow 
pipe, carries 

In [ ]:
# Exercise 2
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

#
try:
    api_key = userdata.get('OPENAI_API_KEY')
    os.environ["OPENAI_API_KEY"] = api_key
    print("✅ \n")
except userdata.SecretNotFoundError:
    print("❌ ")

#
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

def ask_model_direct(question: str) -> str:
    prompt = f"Answer this question:\n{question}\n\nAnswer:"
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

#
QUERIES_MODEL_T = [
    "How do I adjust the carburetor on a Model T?",
    "What is the correct spark plug gap for a Model T Ford?",
    "How do I fix a slipping transmission band?",
    "What oil should I use in a Model T engine?",
]

QUERIES_CR = [
    "What did Mr. Flood have to say about Mayor David Black in Congress on January 13, 2026?",
    "What mistake did Elise Stefanik make in Congress on January 23, 2026?",
    "What is the purpose of the Main Street Parity Act?",
    "Who in Congress has spoken for and against funding of pregnancy centers?",
]

print("="*60)
print("Model T Ford Queries")
print("="*60)
for q in QUERIES_MODEL_T:
    print(f"\nQuestion: {q}")
    print("-" * 40)
    print(ask_model_direct(q))

print("\n" + "="*60)
print("Congressional Record Queries")
print("="*60)
for q in QUERIES_CR:
    print(f"\nQuestion: {q}")
    print("-" * 40)
    print(ask_model_direct(q))

✅ 

Model T Ford Queries

Question: How do I adjust the carburetor on a Model T?
----------------------------------------
Adjusting the carburetor on a Model T Ford involves a few steps to ensure proper fuel-air mixture and engine performance. Here’s a general guide to help you with the adjustment:

1. **Warm Up the Engine**: Start the engine and let it warm up to operating temperature. This helps in making accurate adjustments.

2. **Locate the Carburetor**: The Model T typically has a Holley or Kingston carburetor. Familiarize yourself with its components, including the mixture adjustment screw and the throttle.

3. **Adjust the Mixture**: 
   - Locate the mixture adjustment screw, usually found on the side of the carburetor.
   - Start with the screw turned in (clockwise) until it is lightly seated, then back it out (counterclockwise) about 1.5 to 2 turns. This is a good starting point.
   - While the engine is running, slowly turn the screw in or out to find the point where the eng

In [33]:
# Exercise 6: Query Phrasing Sensitivity
phrasings = {
    "Formal": "What is the recommended maintenance schedule for the engine?",
    "Casual": "How often should I service the engine?",
    "Keywords only": "engine maintenance intervals",
    "Question form": "When do I need to check the engine?",
    "Indirect": "Preventive maintenance requirements"
}

if index.ntotal > 0:
    print("="*60)
    print("Exercise 6: Query Phrasing Sensitivity")
    print("="*60)

    retrieved_sets = {}

    # Retrieve and record top 5 chunks for each phrasing
    for style, query in phrasings.items():
        print(f"\n[{style}] Query: {query}")
        print("-" * 60)

        results = retrieve(query, top_k=5)
        chunk_indices = set()

        for i, (chunk, score) in enumerate(results, 1):
            chunk_indices.add(chunk.chunk_index)
            clean_text = chunk.text[:80].replace('\n', ' ')
            print(f"Top {i} | Score: {score:.4f} | Chunk ID: {chunk.chunk_index:^4} | Text: {clean_text}...")

        retrieved_sets[style] = chunk_indices

    # Compare
    print("\n" + "="*60)
    print("Overlap Analysis (Shared Chunk IDs)")
    print("="*60)

    styles = list(phrasings.keys())
    for i in range(len(styles)):
        for j in range(i+1, len(styles)):
            s1, s2 = styles[i], styles[j]
            overlap = retrieved_sets[s1].intersection(retrieved_sets[s2])
            print(f"{s1} vs {s2}: {len(overlap)} shared chunks -> {list(overlap)}")
else:
    print("Please complete the pipeline setup first.")

Exercise 6: Query Phrasing Sensitivity

[Formal] Query: What is the recommended maintenance schedule for the engine?
------------------------------------------------------------
Top 1 | Score: 0.4790 | Chunk ID:  54  | Text: ew car the following points of care are of particular  importance: Do not drive ...
Top 2 | Score: 0.4486 | Chunk ID:  55  | Text: as soon as possible after need for them  is noticed.  Drain and refill radiator ...
Top 3 | Score: 0.4155 | Chunk ID:  73  | Text: ng  each section once a week. A letter should be written by the service  manager...
Top 4 | Score: 0.3982 | Chunk ID:  53  | Text: ade machinery requires a certain amount of attention, if  it is to continue to o...
Top 5 | Score: 0.3952 | Chunk ID: 524  | Text: time. To insure maximum power, together with  minimum oil and gas consumption, t...

[Casual] Query: How often should I service the engine?
------------------------------------------------------------
Top 1 | Score: 0.5085 | Chunk ID:  73  | Text: ng 

In [34]:
# Exercise 7: Chunk Overlap Experiment

import time

overlaps_to_test = [0, 64, 128, 256]
fixed_chunk_size = 512
cross_boundary_query = "What are the steps to remove the top tank top using the first method?"

print("="*60)
print("Exercise 7: Testing Chunk Overlap (Size=512)")
print("="*60)

for overlap in overlaps_to_test:
    print(f"\n\n{'*'*60}")
    print(f"Rebuilding Index | Size: {fixed_chunk_size} | Overlap: {overlap}")
    print(f"{'*'*60}")

    start_time = time.time()

    rebuild_pipeline(chunk_size=fixed_chunk_size, chunk_overlap=overlap)

    rebuild_time = time.time() - start_time
    print(f"Time taken: {rebuild_time:.2f}s | Total Chunks: {len(all_chunks)}")

    #
    print(f"\nQuery: {cross_boundary_query}")
    print("-" * 40)

    #
    answer = rag_query(cross_boundary_query, top_k=3, show_context=True)

    print("\nAnswer:")
    print(answer)

Exercise 7: Testing Chunk Overlap (Size=512)


************************************************************
Rebuilding Index | Size: 512 | Overlap: 0
************************************************************


Batches:   0%|          | 0/34 [00:00<?, ?it/s]

Rebuilt: 1058 chunks, chunk_size=512, chunk_overlap=0
Time taken: 3.37s | Total Chunks: 1058

Query: What are the steps to remove the top tank top using the first method?
----------------------------------------
RETRIEVED CONTEXT:
[Source: ModelTNew.pdf, Relevance: 0.817]
828. 
Removing Top Tank Top 
833 
The top tank top can be removed in two ways. The first method 
is used when the old top tank top is 0. K. and is to be used again. 
The second method is used when the top section of the tank is beyond 
repair. 
First Method 
834 
To remove the top by the first method, remove the wall as de-
scribed in Pars. 824, 825 or 829. 
835 
Remove the splash plate and the overflow pipe as described in 
Par. 843.

---

[Source: ModelTNew.pdf, Relevance: 0.773]
[Page 216]
198 
FORD SERVICE 
Fig. 418 
Second Method 
836 To remove the top tank top by the second method, remove the 
front or rear wall as described in P ars. 824 or 829. Cut through 
the top with a hack-saw about one inch above the top 

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

Rebuilt: 1239 chunks, chunk_size=512, chunk_overlap=64
Time taken: 2.72s | Total Chunks: 1239

Query: What are the steps to remove the top tank top using the first method?
----------------------------------------
RETRIEVED CONTEXT:
[Source: ModelTNew.pdf, Relevance: 0.750]
ther side. It is 
then a simple matter to drive the top off. 


[Page 216]
198 
FORD SERVICE 
Fig. 418 
Second Method 
836 To remove the top tank top by the second method, remove the 
front or rear wall as described in P ars. 824 or 829. Cut through 
the top with a hack-saw about one inch above the top and header 
joint, stopping at the flange of the remaining wall (See Fig. 418). 
Heat the joint between the top and the wall and brush off the solder 
as it flows out.

---

[Source: ModelTNew.pdf, Relevance: 0.747]
sed when the old top tank top is 0. K. and is to be used again. 
The second method is used when the top section of the tank is beyond 
repair. 
First Method 
834 
To remove the top by the first method, remo

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Rebuilt: 1496 chunks, chunk_size=512, chunk_overlap=128
Time taken: 3.01s | Total Chunks: 1496

Query: What are the steps to remove the top tank top using the first method?
----------------------------------------
RETRIEVED CONTEXT:
[Source: ModelTNew.pdf, Relevance: 0.778]
e pliers with a 
hammer. 
When one side has started, start the other side. It is 
then a simple matter to drive the top off. 


[Page 216]
198 
FORD SERVICE 
Fig. 418 
Second Method 
836 To remove the top tank top by the second method, remove the 
front or rear wall as described in P ars. 824 or 829. Cut through 
the top with a hack-saw about one inch above the top and header 
joint, stopping at the flange of the remaining wall (See Fig. 418).

---

[Source: ModelTNew.pdf, Relevance: 0.755]
cessary use the dent puller as explained in Par. 828. 
Removing Top Tank Top 
833 
The top tank top can be removed in two ways. The first method 
is used when the old top tank top is 0. K. and is to be used again. 
The second met

Batches:   0%|          | 0/75 [00:00<?, ?it/s]

Rebuilt: 2397 chunks, chunk_size=512, chunk_overlap=256
Time taken: 4.81s | Total Chunks: 2397

Query: What are the steps to remove the top tank top using the first method?
----------------------------------------
RETRIEVED CONTEXT:
[Source: ModelTNew.pdf, Relevance: 0.822]
when the old top tank top is 0. K. and is to be used again. 
The second method is used when the top section of the tank is beyond 
repair. 
First Method 
834 
To remove the top by the first method, remove the wall as de-
scribed in Pars. 824, 825 or 829. 
835 
Remove the splash plate and the overflow pipe as described in 
Par. 843. 
Heat the joints between the top header and wall and 
brush off the solder as it runs out.

---

[Source: ModelTNew.pdf, Relevance: 0.802]
le 
and grasp the open end of the top with a pair of pliers (See Fig. 417). 
After heating the joints with a flood flame, tap the pliers with a 
hammer. 
When one side has started, start the other side. It is 
then a simple matter to drive the top off.

In [35]:
# Exercise 8: Chunk Size Experiment

import time


chunk_sizes_to_test = [128, 512, 2048]
fixed_overlap = 20

test_queries = [
    "What is the correct spark plug gap for a Model T Ford?",            # Specific fact
    "How do I adjust the carburetor?",                                   # Short process
    "What are the steps to remove the top tank top using the first method?", # Long process
    "What oil should I use in the engine?",                              # Fact/Recommendation
    "What is the recommended maintenance schedule?"                      # General concept
]

print("="*60)
print("Exercise 8: Testing Chunk Size (128, 512, 2048)")
print("="*60)

for size in chunk_sizes_to_test:
    print(f"\n\n{'*'*60}")
    print(f"Rebuilding Index | Chunk Size: {size} | Overlap: {fixed_overlap}")
    print(f"{'*'*60}")

    start_time = time.time()
    # Rebuild index with the current chunk size
    rebuild_pipeline(chunk_size=size, chunk_overlap=fixed_overlap)
    rebuild_time = time.time() - start_time

    print(f"Time taken: {rebuild_time:.2f}s | Total Chunks: {len(all_chunks)}")

    # Run the 5 queries
    for i, q in enumerate(test_queries, 1):
        print(f"\n--- Query {i}: {q} ---")
        # Set show_context=False to keep output clean, but you can change it if needed
        answer = rag_query(q, top_k=3, show_context=False)
        # Truncate answer for cleaner output display
        print(f"Answer: {answer[:300]}...\n")

Exercise 8: Testing Chunk Size (128, 512, 2048)


************************************************************
Rebuilding Index | Chunk Size: 128 | Overlap: 20
************************************************************


Batches:   0%|          | 0/160 [00:00<?, ?it/s]

Rebuilt: 5091 chunks, chunk_size=128, chunk_overlap=20
Time taken: 3.02s | Total Chunks: 5091

--- Query 1: What is the correct spark plug gap for a Model T Ford? ---
Answer: The correct spark plug gap for a Model T Ford should be checked before replacing it. According to the context, this involves checking the gap between the points on the spark plug. 

Relevant quote from the context: "Before replacing the plug, check the spark plug points for gap, the gap between the ...


--- Query 2: How do I adjust the carburetor? ---
Answer: To adjust the carburetor, you should first see "A" Fig. 561. Then, insert the carburetor adjusting rod "D" through the instrument board and dash. Next, disconnect the carburetor adjusting rod at the carburetor as shown by "B" Fig. 28. Finally, remove the carburetor adjusting screw mentioned in "C" F...


--- Query 3: What are the steps to remove the top tank top using the first method? ---
Answer: The top tank top can be removed in two ways as mentioned in P

Batches:   0%|          | 0/35 [00:00<?, ?it/s]

Rebuilt: 1107 chunks, chunk_size=512, chunk_overlap=20
Time taken: 2.52s | Total Chunks: 1107

--- Query 1: What is the correct spark plug gap for a Model T Ford? ---
Answer: Based on the information provided in the context, there isn't explicit mention of the exact recommended spark plug gap for a Model T Ford. However, the text does indicate that "When filing ends of rings, care should be taken that ring is not distorted as it is possible in this way to get a larger ga...


--- Query 2: How do I adjust the carburetor? ---
Answer: To adjust the carburetor, you need to follow these steps:

1. Insert the head of the carburetor adjusting rod into the slot in the dashboard.
2. Position the forked end of the rod "C" through the head of the carburetor needle valve, ensuring it locks in place with a cotter key inserted through its e...


--- Query 3: What are the steps to remove the top tank top using the first method? ---
Answer: According to the context, the first method involves removing 

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Rebuilt: 281 chunks, chunk_size=2048, chunk_overlap=20
Time taken: 1.19s | Total Chunks: 281

--- Query 1: What is the correct spark plug gap for a Model T Ford? ---
Answer: According to the context, the correct spark plug gap for a Model T Ford is approximately 2 mm (measured in inches, not millimeters). Specifically, the text states: "Also examine the porcelain to make sure that it has not been cracked." Following this instruction, I'll focus on providing the exact me...


--- Query 2: How do I adjust the carburetor? ---
Answer: To adjust the carburetor, follow these steps:

1. **Adjust the Throttle Lever**: 
   - Insert the two carburetor flange bolts through the inlet pipe.
   - Position the carburetor so that the bolts can be entered through the carburetor flange as shown at "C".
   - Tighten the carburetor flange bolt n...


--- Query 3: What are the steps to remove the top tank top using the first method? ---
Answer: According to the context, the steps to remove the top tank top

In [36]:
# Exercise 9: Retrieval Score Analysis

# 10 queries
queries_for_analysis = [
    "What is the correct spark plug gap?",                 # 1. Specific
    "How do I adjust the carburetor?",                    # 2. Specific process
    "What are the steps to remove the top tank top?",          # 3. Long process
    "What is the recommended maintenance schedule?",            # 4. General
    "What oil should I use in the engine?",                # 5. Recommendation
    "How to repair a flat tire?",                      # 6. Common but maybe missing
    "What is the wheelbase of the Model T?",                # 7. Specific fact
    "Tell me about the steering gear.",                   # 8. Broad topic
    "What is the capital of France?",                    # 9. Completely off-topic
    "Why does the manual recommend synthetic oil?"             # 10. False premise
]

if index.ntotal > 0:
    print("="*80)
    print("PART 1: Score Distribution Analysis (Top 10)")
    print("="*80)

    for i, q in enumerate(queries_for_analysis, 1):
        print(f"\n[Query {i}] {q}")
        results = retrieve(q, top_k=10)

        scores = [round(score, 4) for chunk, score in results]
        gap = scores[0] - scores[1] if len(scores) > 1 else 0

        print(f"  Scores: {scores}")
        print(f"  Gap (#1 - #2): {gap:.4f} | Top Score: {scores[0]:.4f}")

    print("\n\n" + "="*80)
    print("PART 2: Experiment - Implementing a Score Threshold (> 0.5)")
    print("="*80)

    def retrieve_with_threshold(query, top_k=5, threshold=0.5):
        results = retrieve(query, top_k=top_k)
        filtered_results = [(chunk, score) for chunk, score in results if score > threshold]
        return filtered_results

    # Test the threshold
    test_query = "What is the capital of France?"
    print(f"\nTesting Threshold on: '{test_query}'")

    # 1. Without Threshold
    raw_results = retrieve(test_query, top_k=3)
    print(f"  Chunks retrieved WITHOUT threshold: {len(raw_results)}")

    # 2. With Threshold > 0.5
    filtered_results = retrieve_with_threshold(test_query, top_k=3, threshold=0.5)
    print(f"  Chunks retrieved WITH threshold > 0.5: {len(filtered_results)}")

    if len(filtered_results) == 0:
        print("  -> System Action: Prevent LLM call. Return 'No relevant documents found.' directly.")
else:
    print("Please complete the pipeline setup first.")

PART 1: Score Distribution Analysis (Top 10)

[Query 1] What is the correct spark plug gap?
  Scores: [0.4374, 0.4249, 0.4104, 0.3849, 0.3504, 0.3345, 0.3343, 0.331, 0.3305, 0.3302]
  Gap (#1 - #2): 0.0125 | Top Score: 0.4374

[Query 2] How do I adjust the carburetor?
  Scores: [0.6645, 0.6641, 0.5637, 0.5539, 0.5355, 0.4947, 0.4833, 0.4805, 0.4797, 0.4757]
  Gap (#1 - #2): 0.0004 | Top Score: 0.6645

[Query 3] What are the steps to remove the top tank top?
  Scores: [0.7116, 0.6343, 0.5874, 0.5556, 0.5038, 0.4997, 0.4907, 0.4835, 0.4726, 0.4708]
  Gap (#1 - #2): 0.0773 | Top Score: 0.7116

[Query 4] What is the recommended maintenance schedule?
  Scores: [0.4793, 0.4239, 0.3387, 0.3194, 0.3178, 0.3107, 0.2736, 0.2668, 0.2662, 0.2522]
  Gap (#1 - #2): 0.0554 | Top Score: 0.4793

[Query 5] What oil should I use in the engine?
  Scores: [0.3513, 0.3008, 0.2986, 0.2919, 0.2738, 0.2681, 0.2638, 0.2537, 0.2487, 0.2444]
  Gap (#1 - #2): 0.0505 | Top Score: 0.3513

[Query 6] How to repair a f

In [38]:
# Exercise 10: Prompt Template Variations


# 5 prompt
prompts = {
    "Minimal": """
{context}

{question}
""",

    "Strict Grounding": """
Answer ONLY based on the context. If the answer isn't there, say 'I cannot answer this from the available documents.'
CONTEXT:
{context}

QUESTION: {question}
ANSWER:""",

    "Encouraging Citation": """
Answer the question based on the context. Quote the exact passages that support your answer using quotation marks.
CONTEXT:
{context}

QUESTION: {question}
ANSWER:""",

    "Permissive": """
Use the context to help answer the question, but you may also use your own general knowledge if the context is insufficient.
CONTEXT:
{context}

QUESTION: {question}
ANSWER:""",

    "Structured Output": """
First list relevant facts from the context in bullet points, then synthesize your final answer.
CONTEXT:
{context}

QUESTION: {question}

FACTS AND SYNTHESIS:"""
}

# 5 test queries
test_queries = [
    "What is the correct spark plug gap for a Model T Ford?",
    "What are the steps to remove the top tank top using the first method?",
    "What is the recommended maintenance schedule?",
    "Why does the manual recommend synthetic oil?",
    "How to repair a flat tire?"
]

if index.ntotal > 0:
    print("="*80)
    print("Exercise 10: Testing Prompt Variations")
    print("="*80)

    for template_name, prompt_text in prompts.items():
        print(f"\n\n{'#'*80}")
        print(f"PROMPT STYLE: {template_name.upper()}")
        print(f"{'#'*80}")

        for i, q in enumerate(test_queries, 1):
            print(f"\n[Query {i}] {q}")
            print("-" * 40)

            # top_k=3
            answer = rag_query(q, top_k=3, show_context=False, prompt_template=prompt_text)


            print(answer)
else:
    print("Please complete the pipeline setup first.")

Exercise 10: Testing Prompt Variations


################################################################################
🛠️ PROMPT STYLE: MINIMAL
################################################################################

[Query 1] What is the correct spark plug gap for a Model T Ford?
----------------------------------------
To determine the correct spark plug gap for a Model T Ford, we need to follow these steps:

1. **Identify the Correct Terminals**: According to the text, the commutator terminals on the coil box are numbered 1, 2, 3, and 4 to correspond with the cylinders.
   
2. **Check the Points Gap**: The text states that the gap between the points should measure approximately 2 mm (or ~2").

Therefore, the correct spark plug gap for a Model T Ford is **approximately 2 mm**. 

This information is crucial because ensuring the proper gap helps prevent arcing issues, which could lead to misfires or other electrical problems. Proper maintenance includes regularly checking a

In [39]:
# Exercise 11: Cross-Document/Cross-Chunk Synthesis

synthesis_queries = [
    "What are ALL the maintenance tasks, oil changes, or checks I need to perform regularly?", # Task synthesis
    "What specific tools or equipment are mentioned for repairing the radiator and tank?",    # Tool synthesis
    "Summarize any safety warnings, care instructions, or precautions mentioned."          # Warning synthesis
]

k_values = [3, 5, 10]

if index.ntotal > 0:
    print("="*80)
    print("Exercise 11: Cross-Chunk Synthesis Experiment")
    print("="*80)

    for i, q in enumerate(synthesis_queries, 1):
        print(f"\n\n{'#'*80}")
        print(f"SYNTHESIS QUERY {i}: {q}")
        print(f"{'#'*80}")

        for k in k_values:
            print(f"\n--- Testing with top_k = {k} ---")
            # prompt
            prompt = """Synthesize a comprehensive answer based ONLY on the provided context.
List all relevant points you can find. If pieces of information are in different paragraphs, combine them logically.
CONTEXT:
{context}

QUESTION: {question}
ANSWER:"""

            # Show context is False to keep output clean, but you can turn it on if you want to see the raw chunks
            answer = rag_query(q, top_k=k, show_context=False, prompt_template=prompt)
            print(answer)
            print("-" * 60)
else:
    print("Please complete the pipeline setup first.")

Exercise 11: Cross-Chunk Synthesis Experiment


################################################################################
SYNTHESIS QUERY 1: What are ALL the maintenance tasks, oil changes, or checks I need to perform regularly?
################################################################################

--- Testing with top_k = 3 ---
Regular maintenance tasks include changing the oil in the crankcase at the end of the first 400 miles and every 750 miles thereafter, except during cold weather, when it is advisable to change oil every 500 miles. Additionally, keep the car well oiled and greased throughout, have adjustments made as soon as possible after noticing their need, drain and refill the radiator frequently, inspect the battery every two weeks and add fresh distilled water, keep all bolts and nuts properly tightened, and ensure the knowledge and skill of the mechanic are always considered. Furthermore, the dealer should maintain an inventory representing approximately

---
## Save/Load Your Index

For large corpora, you don't want to re-embed every time. Here's how to persist the index.

In [ ]:
import pickle

def save_index(filepath: str):
    """Save FAISS index and chunks to disk."""
    faiss.write_index(index, f"{filepath}.faiss")
    with open(f"{filepath}.chunks", 'wb') as f:
        pickle.dump(all_chunks, f)
    print(f"✓ Saved index to {filepath}.faiss")
    print(f"✓ Saved chunks to {filepath}.chunks")

def load_saved_index(filepath: str):
    """Load FAISS index and chunks from disk."""
    global index, all_chunks
    index = faiss.read_index(f"{filepath}.faiss")
    with open(f"{filepath}.chunks", 'rb') as f:
        all_chunks = pickle.load(f)
    print(f"✓ Loaded index with {index.ntotal} vectors")

# Save your index
if index.ntotal > 0:
    save_index("my_rag_index")
else:
    print("No index to save.")

# Later, to load:
# load_saved_index("my_rag_index")

---
## Next Steps

You've built a complete RAG pipeline from scratch! In the next class, we'll:

1. **Improve retrieval** with query rewriting and hybrid search
2. **Rebuild with LangChain** to see how frameworks abstract these steps
3. **Evaluate systematically** with test questions and metrics

### Exercises to try:
- Vary chunk size (256, 512, 1024) and measure retrieval quality
- Try a different embedding model (`BAAI/bge-small-en-v1.5`)
- Try a larger LLM (`Qwen/Qwen2.5-3B-Instruct`) and compare answer quality
- Ask questions that require combining information from multiple chunks

---
## Appendix: Device Information

Run this cell to see detailed information about your compute environment.

In [ ]:
def print_device_info():
    """Print detailed information about available compute devices."""
    print("=" * 60)
    print("DEVICE INFORMATION")
    print("=" * 60)

    print(f"\nEnvironment: {ENVIRONMENT}")
    print(f"PyTorch version: {torch.__version__}")

    # CUDA
    print(f"\nCUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  Device: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    # MPS
    print(f"\nMPS available: {torch.backends.mps.is_available()}")
    print(f"MPS built: {torch.backends.mps.is_built()}")

    # Current selection
    print(f"\n→ Selected device: {DEVICE}")
    print(f"→ Selected dtype: {DTYPE}")
    print("=" * 60)

print_device_info()

========================================================================================================
